In [ ]:
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.spatial.distance import mahalanobis
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from collections import Counter
import json

# ─────────────────────────────────────────────────────────────────
# 1. Definitions (no nfstream, plain strings)
# ─────────────────────────────────────────────────────────────────
DL_TO_ML_CLASSIFIER = {
    "DDoS": "DDoS",
    "Application-Layer DDoS": "AppDDoS",
    "DRDoS": "DRDoS",
    "Web Attacks": "WebAttack",
    "Exploits & Malware": "Exploits",
    "Brute Force & Recon": "BruteForce",
    "Volumetric_UDP_ICMP": "DDoS",
    "TCP_Protocol_Attack": "DDoS",
    "Application_Layer_DoS": "AppDDoS",
    "Reflection_DRDoS": "DRDoS",
    "Web_Infiltration": "WebAttack",
    "Brute_Force_Credentials": "BruteForce",
}

RL_TO_ML_CLASSIFIER = {
    "DDoS": "DDoS", "DDoS-ACK_Fragmentation": "DDoS", "DDoS-ICMP_Flood": "DDoS",
    "DDoS-ICMP_Fragmentation": "DDoS", "DDoS-PSHACK_Flood": "DDoS",
    "DDoS-RSTFINFlood": "DDoS", "DDoS-SYN_Flood": "DDoS", "DDoS-SlowLoris": "DDoS",
    "DDoS-SynonymousIP_Flood": "DDoS", "DDoS-TCP_Flood": "DDoS",
    "DDoS-UDP_Flood": "DDoS", "DDoS-UDP_Fragmentation": "DDoS",
    "Mirai-greeth_flood": "DDoS", "Mirai-greip_flood": "DDoS", "Mirai-udpplain": "DDoS",
    "DoS_GoldenEye": "AppDDoS", "DoS_Hulk": "AppDDoS", "DoS_Slowhttptest": "AppDDoS",
    "DoS_slowloris": "AppDDoS",
    "DrDoS_DNS": "DRDoS", "DrDoS_MSSQL": "DRDoS", "DrDoS_NetBIOS": "DRDoS",
    "DrDoS_NTP": "DRDoS",
    "FTP-Patator": "BruteForce", "SSH-Patator": "BruteForce",
    "SqlInjection": "WebAttack", "XSS": "WebAttack", "Web_Attack_XSS": "WebAttack",
    "CommandInjection": "WebAttack",
    "PortScan": "BruteForce", "Recon-HostDiscovery": "BruteForce", "Recon-PortScan": "BruteForce",
    "Backdoor_Malware": "Exploits", "Bot": "Exploits", "BrowserHijacking": "Exploits",
    "Shellcode": "Exploits", "Worms": "Exploits", "Heartbleed": "Exploits",
    "TFTP": "Exploits", "DNS_Spoofing": "Exploits",
    "Generic": None, "Benign": "Benign",
}

MAIN_CATEGORIES = ["DDoS", "AppDDoS", "DRDoS", "WebAttack", "Exploits", "BruteForce", "Benign"]
ML_CONFIDENCE_THRESHOLD = 0.7
BLACKLISTED_LABELS = {"MITM-ArpSpoofing", "MITM_ArpSpoofing", "mitm-arpspoofing", "Generic"}

def is_blacklisted(label):
    return label.lower() in {l.lower() for l in BLACKLISTED_LABELS}

def _load_label_map(raw_map):
    first_key = next(iter(raw_map))
    if first_key.lstrip('-').isdigit():
        return {int(k): v for k, v in raw_map.items()}
    else:
        return {int(v): k for k, v in raw_map.items()}

def _align_features(df, expected_features, model_name="Model"):
    expected = list(expected_features)
    missing = [f for f in expected if f not in df.columns]
    extra   = [f for f in df.columns if f not in expected]
    if missing:
        for f in missing:
            df[f] = 0
    if extra:
        df = df.drop(columns=extra)
    return df[expected]

# Model architectures
class ImprovedAutoencoder(nn.Module):
    def __init__(self, input_dim=88):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(88,512),nn.BatchNorm1d(512),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(512,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(128,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,16),nn.BatchNorm1d(16),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(16,16))
        self.decoder = nn.Sequential(
            nn.Linear(16,16),nn.BatchNorm1d(16),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(16,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(128,512),nn.BatchNorm1d(512),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(512,88))
    def forward(self,x): return self.decoder(self.encoder(x))

class RoutingNet(nn.Module):
    def __init__(self, input_dim, num_experts):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_dim,256),nn.LayerNorm(256),nn.Dropout(0.1),nn.ReLU(),
            nn.Linear(256,256),nn.LayerNorm(256),nn.ReLU(),
            nn.Linear(256,256),nn.LayerNorm(256),nn.ReLU(),
            nn.Linear(256,128),nn.LayerNorm(128),nn.ReLU(),
            nn.Linear(128,256),nn.LayerNorm(256),nn.ReLU(),
            nn.Linear(256,num_experts))
    def forward(self,x): return self.sequential(x)

class Expert(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim,512),nn.LayerNorm(512),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(512,128),nn.LayerNorm(128),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(128,32),nn.LayerNorm(32),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(32,32),nn.LayerNorm(32),nn.ReLU(),
            nn.Linear(32,32),nn.LayerNorm(32),nn.ReLU(),
            nn.Linear(32,16),nn.LayerNorm(16),nn.ReLU(),
            nn.Linear(16,8))
        self.classifier = nn.Sequential(
            nn.Linear(8,64),nn.LayerNorm(64),nn.ReLU(),nn.Dropout(0.01),
            nn.Linear(64,128),nn.LayerNorm(128),nn.ReLU(),
            nn.Linear(128,num_classes))
    def forward(self,x): return self.classifier(self.encoder(x))

# Wrappers (ML_IDS, DL_IDS, RL_IDS) – identical to deployment
class ML_IDS:
    _ML_IDS_CONFIG = [
        {"name":"AppDDoS","folder":"application_layer_ddos_attacks","model_file":"xgboost_application_ddos_model_nfstream.pkl","features_file":"xgboost_application_ddos_features_nfstream.pkl","scaler_file":"minmax_scaler_application_ddos_attack_nfstream.pkl","label_map_file":"application_ddos_label_mapping.json"},
        {"name":"BruteForce","folder":"bruteforce_and_reconsense","model_file":"xgboost_application_ddos_model_nfstream.pkl","features_file":"xgboost_application_ddos_features_nfstream.pkl","scaler_file":"minmax_scaler_application_ddos_attack_nfstream.pkl","label_map_file":"application_ddos_label_mapping.json"},
        {"name":"DDoS","folder":"ddos","model_file":"xgboost_ddos_model_nfstream.pkl","features_file":"xgboost_ddos_features_nfstream.pkl","scaler_file":"minmax_scaler_ddos_attack_nfstream.pkl","label_map_file":"ddos_label_mapping.json"},
        {"name":"Exploits","folder":"exploits_and_valnurabilties_and_malware","model_file":"xgboost_malware_model_nfstream.pkl","features_file":"xgboost_malware_features_nfstream.pkl","scaler_file":"minmax_scaler_application_malware_attack_nfstream.pkl","label_map_file":"malware_label_mapping.json"},
        {"name":"DRDoS","folder":"Reflection_DRDoS","model_file":"xgboost_drdos_model_nfstream.pkl","features_file":"xgboost_drdos_features_nfstream.pkl","scaler_file":"minmax_scaler_drdos_attack_nfstream.pkl","label_map_file":"drdos_label_mapping.json"},
        {"name":"WebAttack","folder":"web_attack","model_file":"xgboost_web_model_nfstream.pkl","features_file":"xgboost_web_features_nfstream.pkl","scaler_file":"minmax_scaler_web_attack_nfstream.pkl","label_map_file":"web_label_mapping.json"},
    ]

    def __init__(self, model_dir):
        self.classifiers = []
        for cfg in self._ML_IDS_CONFIG:
            folder = model_dir + "\\" + cfg["folder"]
            try:
                model = joblib.load(folder + "\\" + cfg["model_file"])
                features = joblib.load(folder + "\\" + cfg["features_file"])
                scaler = joblib.load(folder + "\\" + cfg["scaler_file"])
                with open(folder + "\\" + cfg["label_map_file"]) as f:
                    label_map = _load_label_map(json.load(f))
                blacklisted = []
                filtered = {}
                for idx, lbl in label_map.items():
                    if is_blacklisted(lbl):
                        blacklisted.append(idx)
                    else:
                        filtered[idx] = lbl
                if hasattr(model, "feature_names_in_"):
                    fnames = [str(f) for f in model.feature_names_in_]
                elif isinstance(features, list):
                    fnames = [f for f in features if f not in ("label","id","category")]
                else:
                    fnames = [f for f in list(features) if f not in ("label","id","category")]
                self.classifiers.append({
                    "name": cfg["name"],
                    "model": model,
                    "scaler": scaler,
                    "features": fnames,
                    "label_map": filtered,
                    "blacklisted": blacklisted
                })
                print(f"[ML-IDS] Loaded {cfg['name']}: {len(fnames)} features, {len(filtered)} classes")
            except Exception as e:
                print(f"[ML-IDS ERROR] {cfg['name']}: {e}")

    def predict_per_flow(self, df):
        n_flows = len(df)
        results = [{"label": "Benign", "conf": 0.0, "classifier": "Benign"} for _ in range(n_flows)]
        for clf in self.classifiers:
            model, scaler, features, label_map = clf["model"], clf["scaler"], clf["features"], clf["label_map"]
            blacklisted = clf.get("blacklisted", [])
            sfeats = list(scaler.feature_names_in_) if hasattr(scaler, "feature_names_in_") else features
            aligned = _align_features(df.copy(), sfeats)
            try:
                scaled = scaler.transform(aligned)
            except:
                continue
            X = _align_features(pd.DataFrame(scaled, columns=sfeats), features).to_numpy()
            try:
                preds = model.predict(X)
                probs = model.predict_proba(X)
            except:
                continue
            benign_idx = next((k for k, v in label_map.items() if v.lower() == "benign"), 0)
            for i in range(n_flows):
                pred = int(preds[i])
                conf = float(probs[i].max())
                if pred in blacklisted:
                    pred = benign_idx
                label = label_map.get(pred, "Benign")
                if label.lower() != "benign" and conf > results[i]["conf"] and conf >= ML_CONFIDENCE_THRESHOLD:
                    results[i] = {"label": label, "conf": conf, "classifier": clf["name"]}
        return results


class DL_IDS:
    def __init__(self, model_dir, ae_percentile=94):
        self.model = ImprovedAutoencoder()
        self.model.load_state_dict(torch.load(model_dir + "\\mlp_best_numeric_autoencoder.pth",
                                              map_location="cpu", weights_only=True))
        self.model.eval()
        self.threshold = float(np.percentile(joblib.load(model_dir + "\\all_error"), ae_percentile))
        self.scaler = joblib.load(model_dir + "\\#base_MLP_scaler.pkl")
        self.features = self.scaler.feature_names_in_
        self.prototypes = joblib.load(model_dir + "\\prototypes.pkl")
        print(f"[DL-IDS] Loaded. Threshold={self.threshold:.6f}")

    def _get_latent(self, x):
        with torch.no_grad():
            return self.model.encoder(x).cpu().numpy()

    def _reconstruct_mse(self, x):
        with torch.no_grad():
            return torch.mean((self.model(x) - x) ** 2, dim=1).cpu().numpy()

    def _mahalanobis_label(self, latent_vec):
        distances = {}
        for cat, stats in self.prototypes.items():
            try:
                d = mahalanobis(latent_vec, stats["mean"], stats["cov_inv"])
            except:
                d = float(np.linalg.norm(latent_vec - stats["mean"]))
            distances[cat] = d
        return min(distances, key=distances.get)

    def predict_per_flow(self, df):
        aligned = _align_features(df.copy(), self.features)
        try:
            scaled = self.scaler.transform(aligned)
        except:
            return [{"label": "Benign", "anomalous": False, "mse": 0.0, "classifier": "Benign"} for _ in range(len(df))]
        x = torch.tensor(scaled, dtype=torch.float32)
        mse = self._reconstruct_mse(x)
        latent = self._get_latent(x)
        results = []
        for i in range(len(df)):
            if mse[i] > self.threshold:
                dl_label = self._mahalanobis_label(latent[i])
                ml_clf = DL_TO_ML_CLASSIFIER.get(dl_label, dl_label)
                results.append({
                    "label": dl_label,
                    "classifier": ml_clf,
                    "anomalous": True,
                    "mse": float(mse[i])
                })
            else:
                results.append({
                    "label": "Benign",
                    "classifier": "Benign",
                    "anomalous": False,
                    "mse": float(mse[i])
                })
        return results


class RL_IDS:
    def __init__(self, model_dir):
        checkpoint = torch.load(model_dir + "\\best_model.pth", map_location="cpu", weights_only=True)
        self.scaler = joblib.load(model_dir + "\\RL_scaler.pkl")
        self.features = self.scaler.feature_names_in_
        with open(model_dir + "\\label_map.json") as f:
            self.label_map = _load_label_map(json.load(f))
        self.blacklisted = {idx for idx, lbl in self.label_map.items() if is_blacklisted(lbl)}
        n_experts = len(checkpoint["experts"])
        num_classes = len(self.label_map)
        self.router = RoutingNet(self.scaler.n_features_in_, n_experts)
        self.router.load_state_dict(checkpoint["router"])
        self.router.eval()
        self.experts = [Expert(self.scaler.n_features_in_, num_classes) for _ in range(n_experts)]
        for i, e in enumerate(self.experts):
            e.load_state_dict(checkpoint["experts"][i])
            e.eval()
        print(f"[RL-IDS] Loaded. Experts={n_experts}, Classes={num_classes}")

    def predict_per_flow(self, df):
        aligned = _align_features(df.copy(), self.features)
        try:
            scaled = self.scaler.transform(aligned)
        except:
            return [{"label": "Benign", "conf": 1.0, "classifier": "Benign"} for _ in range(len(df))]
        x = torch.tensor(scaled, dtype=torch.float32)
        with torch.no_grad():
            router_probs = torch.softmax(self.router(x), dim=1)
            expert_outputs = torch.stack([torch.softmax(e(x), dim=1) for e in self.experts])
            combined = (router_probs.unsqueeze(2) * expert_outputs.permute(1,0,2)).sum(dim=1)
            preds = torch.argmax(combined, dim=1).cpu().numpy()
            confs = combined.max(dim=1).values.cpu().numpy()
        results = []
        for i in range(len(df)):
            pred = int(preds[i])
            if pred in self.blacklisted:
                results.append({"label": "Benign", "conf": float(confs[i]), "classifier": "Benign"})
            else:
                label = self.label_map.get(pred, "Benign")
                clf = RL_TO_ML_CLASSIFIER.get(label, label)
                if clf is None:
                    clf = "Benign"
                results.append({"label": label, "conf": float(confs[i]), "classifier": clf})
        return results


# ─────────────────────────────────────────────────────────────────
# 3. Evaluation with corrected voting rule (Option A)
# ─────────────────────────────────────────────────────────────────
x_res = joblib.load("x_res.pkl")
y_res = joblib.load("y_res.pkl")   # integer class indices

np.random.seed(42)
mask = np.random.rand(len(x_res)) < 0.2
x_sample = x_res[mask]
y_true_int = y_res[mask].astype(int)

# Use the same label map as RL-IDS to convert integer → name
BASE = r"C:\Users\youss\Downloads\uni_stuff\grad_cs_part\full_ids_depolyment(ids_engine)\Models"
with open(BASE + "\\rl_ids\\label_map.json") as f:
    int_to_name = _load_label_map(json.load(f))   # e.g. {0:"Benign", 1:"DDoS", ...}

# Map integer true labels → coarse categories via RL mapping
y_true_str = y_true_int.map(int_to_name).fillna("Benign")
y_true = y_true_str.map(lambda name: RL_TO_ML_CLASSIFIER.get(name, name if name == "Benign" else "Benign"))
y_true = y_true.apply(lambda x: x if x in MAIN_CATEGORIES else "Benign")

print(f"Evaluating on {len(x_sample)} samples (20% of {len(x_res)})")
print("True label distribution:\n", y_true.value_counts())

# Load models
ml_ids = ML_IDS(BASE + "\\ml_ids")
dl_ids = DL_IDS(BASE + "\\dl_ids", ae_percentile=94)
rl_ids = RL_IDS(BASE + "\\rl_ids")

def ml_to_main(pred):
    return pred["classifier"]

def dl_to_main(pred):
    raw = pred["label"]
    return DL_TO_ML_CLASSIFIER.get(raw, raw if raw == "Benign" else "Benign")

def rl_to_main(pred):
    raw = pred["label"]
    mapped = RL_TO_ML_CLASSIFIER.get(raw, raw)
    if mapped is None or mapped == "Benign":
        return "Benign"
    return mapped

# Predict
ml_preds = ml_ids.predict_per_flow(x_sample)
dl_preds = dl_ids.predict_per_flow(x_sample)
rl_preds = rl_ids.predict_per_flow(x_sample)

ml_main = [ml_to_main(p) for p in ml_preds]
dl_main = [dl_to_main(p) for p in dl_preds]
rl_main = [rl_to_main(p) for p in rl_preds]

# ---- CORRECTED VOTING ----
y_vote = []
for ml, dl, rl in zip(ml_main, dl_main, rl_main):
    # DL only casts a vote if it is non‑Benign AND agrees with ML
    dl_vote = dl if (dl != "Benign" and dl == ml) else "Benign"
    # RL always votes its own (Benign or attack)
    rl_vote = rl

    # Collect votes that are non‑Benign
    votes = [v for v in [ml, dl_vote, rl_vote] if v != "Benign"]
    if len(votes) >= 2:
        # At least two non‑Benign votes -> majority
        winner = Counter(votes).most_common(1)[0][0]
    else:
        # Not enough agreement -> fall back to ML's original prediction
        winner = ml   # ml can be Benign or an attack
    y_vote.append(winner)

# Metrics
acc = accuracy_score(y_true, y_vote)
f1  = f1_score(y_true, y_vote, labels=MAIN_CATEGORIES, average='macro', zero_division=0)
cm  = confusion_matrix(y_true, y_vote, labels=MAIN_CATEGORIES)

print(f"\nMajority Vote Results (≥2/3, DL supports ML only when aligned):")
print(f"Accuracy : {acc:.4f}")
print(f"Macro F1 : {f1:.4f}")
print("\nConfusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm, index=MAIN_CATEGORIES, columns=MAIN_CATEGORIES))
print("\nClassification Report:")
print(classification_report(y_true, y_vote, labels=MAIN_CATEGORIES, zero_division=0))

Evaluating on 320339 samples (20% of 1605400)
True label distribution:
 label
DDoS          96075
Exploits      63942
Benign        51313
BruteForce    32173
WebAttack     25717
AppDDoS       25607
DRDoS         25512
Name: count, dtype: int64
[ML-IDS] Loaded AppDDoS: 12 features, 6 classes
[ML-IDS] Loaded BruteForce: 41 features, 5 classes
[ML-IDS] Loaded DDoS: 9 features, 11 classes
[ML-IDS] Loaded Exploits: 9 features, 7 classes
[ML-IDS] Loaded DRDoS: 8 features, 5 classes
